# Notebook 4: Feature Engineering

## Objective

Create engineered features for tire degradation prediction.

The engineered dataset will be used to train the baseline machine learning model.

In [1]:
import pandas as pd
import numpy as np

In [2]:
master_df = pd.read_csv("../data/processed/master_dataset.csv")
master_df.head()

,Driver,Team,LapNumber,Compound,TyreLife,Stint,Position,TrackStatus,LapTimeSeconds
0,VER,Red Bull Racing,1.0,SOFT,4.0,1.0,2.0,1,83.186
1,VER,Red Bull Racing,2.0,SOFT,5.0,1.0,2.0,1,79.871
2,VER,Red Bull Racing,3.0,SOFT,6.0,1.0,1.0,1,79.364
3,VER,Red Bull Racing,4.0,SOFT,7.0,1.0,1.0,1,80.766
4,VER,Red Bull Racing,5.0,SOFT,8.0,1.0,1.0,1,80.827


In [3]:
# checking the dataset information


print(master_df.info())
print("\n Dataset shape:", master_df.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11880 entries, 0 to 11879
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Driver          11880 non-null  object 
 1   Team            11880 non-null  object 
 2   LapNumber       11880 non-null  float64
 3   Compound        11880 non-null  object 
 4   TyreLife        11880 non-null  float64
 5   Stint           11880 non-null  float64
 6   Position        11880 non-null  float64
 7   TrackStatus     11880 non-null  int64  
 8   LapTimeSeconds  11880 non-null  float64
dtypes: float64(5), int64(1), object(3)
memory usage: 835.4+ KB
None

 Dataset shape: (11880, 9)


In [4]:
# fuel load approximation
# because car starts with highest fuel and fuel decreses almost every lap

max_lap = master_df["LapNumber"].max()
master_df["FuelLoadApprox"] = (
    1 - (master_df["LapNumber"] / max_lap)
)
master_df[
    ["LapNumber", "FuelLoadApprox"]
].head()

,LapNumber,FuelLoadApprox
0,1.0,0.984848
1,2.0,0.969697
2,3.0,0.954545
3,4.0,0.939394
4,5.0,0.924242


In [8]:
# feature 2 - stint progress

max_tyrelife = master_df.groupby(
    ["Driver", "Stint"]
)["TyreLife"].transform("max")

# Calculate normalized stint progress
master_df["StintProgress"] = (
    master_df["TyreLife"] / max_tyrelife
)

# Preview
master_df[
    ["Driver", "Stint", "TyreLife", "StintProgress"]
].head(10)

,Driver,Stint,TyreLife,StintProgress
0,VER,1.0,4.0,0.100
1,VER,1.0,5.0,0.125
2,VER,1.0,6.0,0.150
3,VER,1.0,7.0,0.175
4,VER,1.0,8.0,0.200
5,VER,1.0,9.0,0.225
6,VER,1.0,10.0,0.250
7,VER,1.0,11.0,0.275
8,VER,1.0,12.0,0.300
9,VER,1.0,13.0,0.325


In [9]:
# feature 3 - compound encoding
# we need to convert soft,medium,hard into 0,1,2

compound_map = {
    "SOFT": 0,
    "MEDIUM": 1,
    "HARD": 2
}
master_df["CompoundEncoded"] = master_df["Compound"].map(compound_map)
master_df[
    ["Compound", "CompoundEncoded"]
].drop_duplicates()



,Compound,CompoundEncoded
0,SOFT,0
17,MEDIUM,1
234,HARD,2


In [10]:
# feature 4 - fresh tyre indicator
# 1 means brand new tyre whereas 0 means already been used

master_df["FreshTyre"] = (
    master_df["TyreLife"] == 1
).astype(int)

master_df[
    ["TyreLife", "FreshTyre"]
].head(15)

,TyreLife,FreshTyre
0,4.0,0
1,5.0,0
2,6.0,0
3,7.0,0
4,8.0,0
5,9.0,0
6,10.0,0
7,11.0,0
8,12.0,0
9,13.0,0


In [11]:
master_df[master_df["FreshTyre"] == 1][
    ["Driver", "LapNumber", "TyreLife", "FreshTyre", "Stint"]
].head(15)

,Driver,LapNumber,TyreLife,FreshTyre,Stint
17,VER,18.0,1.0,1,2.0
44,VER,45.0,1.0,1,3.0
66,NOR,1.0,1.0,1,1.0
89,NOR,24.0,1.0,1,2.0
148,HAM,17.0,1.0,1,2.0
213,RUS,16.0,1.0,1,2.0
234,RUS,37.0,1.0,1,3.0
264,LEC,1.0,1.0,1,1.0
288,LEC,25.0,1.0,1,2.0
330,SAI,1.0,1.0,1,1.0


In [12]:
# feature 5 - tyre age sqaured

master_df["TyreAgeSquared"] = master_df["TyreLife"] ** 2

master_df[
    ["TyreLife", "TyreAgeSquared"]
].head(10)

,TyreLife,TyreAgeSquared
0,4.0,16.0
1,5.0,25.0
2,6.0,36.0
3,7.0,49.0
4,8.0,64.0
5,9.0,81.0
6,10.0,100.0
7,11.0,121.0
8,12.0,144.0
9,13.0,169.0


In [13]:
# Save feature engineered dataset

master_df.to_csv(
    "../data/processed/master_dataset.csv",
    index=False
)

print("Feature engineered dataset saved successfully!")

Feature engineered dataset saved successfully!
